# EXECUTION SCRIPT

The execution code is the one that actually makes predictions with the model that is already trained.

In technical terms, there is a .predict (prediction) and not a .fit (training).

IMPORTANT: This  code must be executed in the exact same environment in which it was originally created.

The enviroment can be installed on a new machine using the *riesgos.yml* file created (or activated) during the project setup.

Copy the file *riesgos.yml* to your working directory and run this command in the terminal (or Anaconda Prompt): 

*conda env create --file riesgos.yml --name riesgos*

## Execution script

In [1]:
import pandas as pd
import cloudpickle

# --- Import data (load validation / new data) ---
PROJECT_PATH = '/Users/rober/cmapss-rul-prediction'
validation_name_file = 'validation_FD001.csv'
VALIDATION_PATH = PROJECT_PATH + '/02_Data/02_Validation/' + validation_name_file
df = pd.read_csv(VALIDATION_PATH)


# --- Restict to selected variables ---
selected_variables = [
    'time_in_cycles',
    'sensor_11',
    'sensor_4',
    'sensor_12',
    'sensor_7',
    'sensor_15',
    'sensor_21',
    'sensor_20'
]
df = df[selected_variables].copy()


# --- Load trained pipeline ---
pipe_execution_name = 'pipe_execution.pickle'
PIPE_EXECUTION_PATH = PROJECT_PATH + '/04_Models/' + pipe_execution_name

with open(PIPE_EXECUTION_PATH, mode='rb') as file:
    pipe_execution = cloudpickle.load(file)


# --- Make predictions ---
predictions = pipe_execution.predict(df)

/var/folders/6c/byy38myn1t50449jbzfjs1_c0000gn/T/ipykernel_2010/1587467407.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


## Checking predictions over validation data

In [2]:
# --- Import data (load validation / new data) ---
PROJECT_PATH = '/Users/rober/cmapss-rul-prediction'
validation_name_file = 'validation_FD001.csv'
VALIDATION_PATH = PROJECT_PATH + '/02_Data/02_Validation/' + validation_name_file
df = pd.read_csv(VALIDATION_PATH)

# --- Combine predictions with unit_number for reference ---
results = pd.DataFrame({
    'unit_number': df['unit_number'],
    'time_in_cycles': df['time_in_cycles'],
    'predicted_RUL': predictions
})

# Order by unit_number and time_in_cycles
results = results.sort_values(by=['unit_number', 'time_in_cycles']).reset_index(drop=True)

# Optional: preview
display(results.head(40))

,unit_number,time_in_cycles,predicted_RUL
0,1,1,197.163756
1,1,2,199.587691
2,1,3,203.835803
3,1,4,212.619552
4,1,5,196.874596
5,1,6,199.770901
6,1,7,199.596074
7,1,8,205.597881
8,1,9,195.022163
9,1,10,189.839692


## Save that resulting dataframe for the Streamlit we'll build

In [3]:
results.to_csv('../../05_Results/predictions_validation_FD001.csv', index=False)